# Part 2C — Transfer Learning: EfficientNetB0 (Experiments A & B)
**Unit:** CIS143-6 Applications of AI  
> Run after `part2b_ann_vs_cnn.ipynb`. Loads CNN Exp A from Drive as reference.

## 0. Setup & Data

In [ ]:
import os, random, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/brain_tumour_mri')
DATA_DIR = Path('/content/brain_tumour_data')

if not DATA_DIR.exists():
    shutil.copytree(DRIVE_DIR / 'dataset', DATA_DIR)
    print('Dataset restored from Drive.')

TRAIN_DIR = DATA_DIR / 'Training'
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
records = []
for cls in CLASSES:
    for img_path in (TRAIN_DIR / cls).glob('*.jpg'):
        records.append({'filepath': str(img_path), 'label': cls})

df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=15, zoom_range=0.10,
    horizontal_flip=True, width_shift_range=0.10, height_shift_range=0.10)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True, seed=42)
val_gen = val_test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
test_gen = val_test_datagen.flow_from_dataframe(
    test_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

CLASS_NAMES = list(train_gen.class_indices.keys())

cw = compute_class_weight('balanced', classes=np.unique(train_df['label']), y=train_df['label'])
class_weight_dict = dict(enumerate(cw))
print('Class mapping:', train_gen.class_indices)

In [ ]:
def plot_history(history, title):
    e = range(1, len(history.history['accuracy']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(e, history.history['accuracy'], label='Train', lw=2)
    ax1.plot(e, history.history['val_accuracy'], label='Val', lw=2, ls='--')
    ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(e, history.history['loss'], label='Train', lw=2)
    ax2.plot(e, history.history['val_loss'], label='Val', lw=2, ls='--')
    ax2.set_title('Loss'); ax2.legend(); ax2.grid(alpha=0.3)
    plt.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()

def evaluate_model(model, gen, model_name):
    gen.reset()
    loss, acc = model.evaluate(gen, verbose=0)
    gen.reset()
    probs = model.predict(gen, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    y_true = gen.classes
    print(f'\n{model_name} — Accuracy: {acc:.4f} | Loss: {loss:.4f}')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASS_NAMES).plot(
        ax=ax, cmap='Blues', values_format='d')
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=12)
    plt.tight_layout(); plt.show()
    return acc, probs, y_true, y_pred

## 1. Transfer Learning Background

**Transfer learning** reuses feature representations learned on a large dataset (ImageNet, 1.4M images, 1000 classes) and applies them to a new task. This is particularly powerful when the target dataset is small relative to the model's capacity.

**EfficientNetB0** (Tan and Le, 2019) uses compound scaling — uniformly scaling network depth, width, and input resolution — achieving state-of-the-art accuracy with far fewer parameters than ResNet or VGG.

**Two-phase strategy:**
1. **Exp A — Feature Extraction**: Freeze all base layers. Only the classifier head is trained. Fast but limited domain adaptation.
2. **Exp B — Fine-Tuning**: Unfreeze top 30 layers after feature extraction. Use a very low learning rate (1e-5) to prevent destroying pretrained weights.

## 2. EfficientNetB0 — Experiment A: Feature Extraction (Frozen Base)

In [ ]:
base_A = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
base_A.trainable = False

inp = tf.keras.Input(shape=(224, 224, 3))
x = base_A(inp, training=False)   # training=False: use stored BN statistics
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(4, activation='softmax')(x)

eff_A = tf.keras.Model(inp, out, name='effnet_frozen')
eff_A.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])

trainable = sum(tf.size(v).numpy() for v in eff_A.trainable_variables)
total = sum(tf.size(v).numpy() for v in eff_A.variables)
print(f'Trainable params: {trainable:,}  |  Total params: {total:,}')

In [ ]:
callbacks_effA = [
    EarlyStopping(patience=6, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(str(DRIVE_DIR / 'effnet_frozen.keras'), save_best_only=True, monitor='val_loss'),
]

hist_effA = eff_A.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25,
    class_weight=class_weight_dict,
    callbacks=callbacks_effA,
    verbose=1,
)
plot_history(hist_effA, 'EfficientNetB0 Exp A — Feature Extraction (Frozen)')

In [ ]:
eff_A = tf.keras.models.load_model(str(DRIVE_DIR / 'effnet_frozen.keras'))
acc_effA, probs_effA, ytrue, ypred_effA = evaluate_model(eff_A, test_gen, 'EfficientNetB0 Exp A')

## 3. EfficientNetB0 — Experiment B: Fine-Tuning Top 30 Layers

Fine-tuning allows the upper layers of the pretrained network to adapt to MRI-specific features. A learning rate of **1e-5** (100× lower than Exp A) prevents catastrophic forgetting of ImageNet features while permitting domain adaptation.

In [ ]:
# Load from Exp A checkpoint and unfreeze top 30 base layers
eff_B = tf.keras.models.load_model(str(DRIVE_DIR / 'effnet_frozen.keras'))

base_B = eff_B.layers[1]   # the EfficientNetB0 base
base_B.trainable = True
for layer in base_B.layers[:-30]:
    layer.trainable = False

eff_B.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

trainable_ft = sum(tf.size(v).numpy() for v in eff_B.trainable_variables)
print(f'Trainable params after unfreeze: {trainable_ft:,}')

In [ ]:
callbacks_effB = [
    EarlyStopping(patience=6, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-8, verbose=1),
    ModelCheckpoint(str(DRIVE_DIR / 'effnet_finetuned.keras'), save_best_only=True, monitor='val_loss'),
]

hist_effB = eff_B.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25,
    class_weight=class_weight_dict,
    callbacks=callbacks_effB,
    verbose=1,
)
plot_history(hist_effB, 'EfficientNetB0 Exp B — Fine-Tuning Top 30 Layers')

In [ ]:
eff_B = tf.keras.models.load_model(str(DRIVE_DIR / 'effnet_finetuned.keras'))
acc_effB, probs_effB, ytrue, ypred_effB = evaluate_model(eff_B, test_gen, 'EfficientNetB0 Exp B')

In [ ]:
# Overlay Exp A vs Exp B validation curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(hist_effA.history['val_accuracy'], label='Exp A (frozen)', lw=2)
ax1.plot(hist_effB.history['val_accuracy'], label='Exp B (fine-tuned)', lw=2, ls='--')
ax1.set_title('Val Accuracy — Exp A vs Exp B'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(hist_effA.history['val_loss'], label='Exp A (frozen)', lw=2)
ax2.plot(hist_effB.history['val_loss'], label='Exp B (fine-tuned)', lw=2, ls='--')
ax2.set_title('Val Loss — Exp A vs Exp B'); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('EfficientNetB0: Frozen vs Fine-Tuned', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Hyperparameter Comparison Table

In [ ]:
# Fill val_acc from history after running all experiments
hp_table = pd.DataFrame([
    {'Config': 'CNN Exp A',          'Model': 'Improved CNN', 'LR': '5e-4',  'Epochs': 30, 'Dropout': 0.4, 'Input': 'Augmented',     'Val Acc': f"{max(histA.history['val_accuracy']):.4f}"},
    {'Config': 'CNN Exp B',          'Model': 'Improved CNN', 'LR': '1e-3',  'Epochs': 30, 'Dropout': 0.3, 'Input': 'Otsu + Aug',    'Val Acc': f"{max(histB.history['val_accuracy']):.4f}"},
    {'Config': 'EfficientNetB0 Exp A','Model': 'EfficientNetB0','LR': '1e-3', 'Epochs': 25, 'Dropout': 0.3, 'Input': 'Augmented',    'Val Acc': f"{max(hist_effA.history['val_accuracy']):.4f}"},
    {'Config': 'EfficientNetB0 Exp B','Model': 'EfficientNetB0','LR': '1e-5', 'Epochs': 25, 'Dropout': 0.3, 'Input': 'Augmented (FT)','Val Acc': f"{max(hist_effB.history['val_accuracy']):.4f}"},
])

print('Hyperparameter Comparison:')
print(hp_table.to_string(index=False))